This notebook reads documents with open HPLC analysis from cruises ocurred in different seas of the world. The objective is to read them all and create a unified file with all the information, gathered in a xarray object. All original HPLC documents are publicly available, and are copied in this project without any modification.

The notebook is divided in parts:
- Reading and preprocessing most HPLC documents that are in "data/raw/hplc_world" folder
- The Maredat dataset (folder that can be found in "data/raw/hplc_world/PANGAEA" in the project) needs a separate reading and preprocess
- Load in-situ data (D_ins)

Once all the documents are read and preprocessed, we unify them (forcing the same naming for variables) into a single xarray document

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import datetime
import re

from pathlib import Path
from difflib import get_close_matches

p_raw = Path('../../../data/raw/hplc_world/')

### Save data
p_pro = Path('../../../data/processed/hplc_world/')
p_pro.mkdir(parents=True, exist_ok=True)

pigment_names = ['chlide_a[mg*m^3]', 'chla[mg*m^3]', 'chlb[mg*m^3]', 'chlc1+c2[mg*m^3]',
       'fucox[mg*m^3]', "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]",
       'diadino[mg*m^3]', 'allox[mg*m^3]', 'diatox[mg*m^3]', 'zeaxan[mg*m^3]',
       'beta_car[mg*m^3]', 'peridinin[mg*m^3]']


pigment_names_short = [['chlide_a'], ['tot_chl_a', 'chl_a', 'chla'], ['tot_chl_b', 'chl_b', 'chlb'], ['chlc1+c2','chl_c1+c2', 'chl_c1c2'] ,
       ['fucox', 'fuco'], ["19'hxfcx", "hex-fuco"], ["19'btfcx", "but-fuco"],
       ['diadino'], ['allox', 'allo'], ['diatox', 'diato'], ['zeaxan', 'zeax', 'zea'],
       ['beta-beta-car', 'beta_car'], ['peridinin', 'perid']]

## Reading hplc analysis in "data/raw/hplc_world" 

These functions will read a document and structure it as a python object

In [2]:
def is_integer(input_str):
    try:
        int(input_str)
        return True
    except Exception as e:
        return False


def is_number(input_str):
    try:
        float(input_str)
        return True
    except Exception as e:
        return False


def TRS_common_read(fname):
    d = {}
    data = []
    with open(fname, 'r') as file:
        read_data = False
        for line in file:
            stringa = line.strip().strip('!/\n@').split('=')
            if len(stringa) == 1 and stringa[0]== '':
                continue
            if len(stringa) == 2:
                field = stringa[0]
                value = stringa[1]
                value = re.sub(r'[\/]', '_', value)
                if 'Station [none]' in field:
                    continue
                if value[-5:] == '[DEG]' or  value[-5:] == '[GMT]':
                    value = value[:-5]
                if field == 'records' or field == 'fields':
                    measures = value.split(',')
                    measures = [me.lower() for me in measures]
                else:
                    d[field] = value
                    if is_number(d[field]):
                        d[field] = float(d[field])
                    # if is_integer(d[field]):
                    #     d[field] = int(d[field])
            if read_data:
                if ',' in stringa[0]:
                    stringa = stringa[0].strip().split(',')
                else:
                    stringa = stringa[0].strip().split()
                stringa = [float(s) if is_number(s) else s for s in stringa]
                # stringa = [int(s) if is_integer(s) else s for s in stringa]
                data.append(stringa)
            if stringa[0] == 'end_header':
                read_data = True
    if len(data) == 0:
        print("Empty file!!!")
        return None
    if len(measures) != len(data[0]):
        print("Careful! Dimensions of measures does not match with data!!!!")

    # return [ d|dict(zip(measures, data_line)) for data_line in data if data_line[1] != -9.99]

    final_dataframe = pd.DataFrame([d|dict(zip(measures, data_line)) for data_line in data])
    if 'lat' not in final_dataframe.columns or 'lon' not in final_dataframe.columns:
        print(f"Not lat or lon variable found")
        final_dataframe['lat'] = float(d['north_latitude'])
        final_dataframe['lon'] = float(d['east_longitude'])
    return final_dataframe


The function appearing below reads searches inside all directories recursively to find files named as 'archive'

In [3]:
def get_all_archives(path):
    archives = []
    for f in path.iterdir():
        if f.name == 'archive':
            archives.append(f)
        elif f.is_dir():
            archives += get_all_archives(f)
    return archives

get_all_archives(p_raw / 'BOWDOIN')

[WindowsPath('../../../data/raw/hplc_world/BOWDOIN/ROESLER/3rivers/archive'),
 WindowsPath('../../../data/raw/hplc_world/BOWDOIN/ROESLER/BOWDOINBUOY/archive'),
 WindowsPath('../../../data/raw/hplc_world/BOWDOIN/ROESLER/PACE_ABSclosure/Florida2017_ABSclosure/archive')]

In [4]:
# All directies in p_raw

In [5]:
datasets_raw = {dir.name: dir for dir in p_raw.iterdir() if dir.is_dir()}

In [6]:
list(datasets_raw.keys())

['BOWDOIN',
 'CALPOLY',
 'COLUMBIA_U',
 'LOV',
 'MAINE',
 'MLI',
 'NASA_GSFC',
 'NOAA_CSC',
 'NOAA_NESDIS',
 'NRL',
 'ODU',
 'OSU',
 'PANGAEA',
 'SIO',
 'UCONN',
 'UCSB',
 'UMASS_D',
 'UMD',
 'URI',
 'USF',
 'WHOI']

In [7]:
for name, dataset_path in datasets_raw.items():
    if name in ['PANGAEA']:
        continue
    print(name)
    archives = get_all_archives(dataset_path)
    df_list = []
    for arch in archives:
        for f in arch.iterdir():
            if f.is_file():
                print(f)
                new_rows_df = TRS_common_read(f)
                if new_rows_df is None:
                    continue
                if 'day' in new_rows_df.columns and 'month' in new_rows_df.columns and 'year' in new_rows_df.columns:
                    new_rows_df['time'] = pd.to_datetime(new_rows_df[['month', 'day', 'year']])
                elif (new_rows_df['start_date']==new_rows_df['end_date']).all():
                    new_rows_df['time'] = new_rows_df['start_date'].apply(lambda x: datetime.datetime.strptime(str(int(x)), "%Y%m%d"))
                else:
                    new_rows_df['time'] = new_rows_df['date'].apply(lambda x: datetime.datetime.strptime(str(int(x)), "%Y%m%d"))
                df_list.append(new_rows_df)

    hplc = pd.concat(df_list, join="inner", ignore_index=True)
    # hplc['Id'] = range(len(hplc.index))
    # hplc = hplc.set_index('Id')
    hplc = hplc.to_xarray().rename({'index': 'Id'})
    path_dir = p_raw / name
    for var in hplc:
        if hplc[var].dtype not in [int, str, np.datetime64.dtype, float] and var!='time':
            hplc[var] = hplc[var].astype(str)
    hplc.to_netcdf(path_dir/'archive.nc')

BOWDOIN
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\3rivers\archive\NASA_3Rivers_HPLC_NNX11AF22G.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2008_HPLC_NNX10A020G.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2009_HPLC_NNX10A020G.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2010_HPLC_NNX10A020G.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2011_HPLC_NNX10A020G_v2.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2012_HPLC_NNX10A020G_v2.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_2014_2015_2016_HPLC.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\BD02_HPLC_SeaBASS_2017.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\BOWDOINBUOY\archive\EcoHab_2013_HPLC_NNX10A020G.sb
..\..\..\data\raw\hplc_world\BOWDOIN\ROESLER\PACE_ABSclosure\Florida2017_ABSclosure\archive\PACE_ABSclosure_hplc_2017.sb
CALPOLY
..\..\..\data\

#### We saved an xarray object named 'archive.nc' for every folder in p_raw.

## Reading hplc analysis from "data/raw/hplc_world/PANGAEA"

In [8]:
PATH_RAW = Path('../../../data/raw/hplc_world/PANGAEA/')

excel_inp = PATH_RAW / 'MAREDAT_pigments_master_file_Updated210313.xls'

nc_out = PATH_RAW / 'archive.nc'

sheet_names = ['MAREDAT_HPLC_PIGMENTS', 'Header_Definitions']

Read excel file

In [9]:

hplc = pd.read_excel(excel_inp, sheet_name='MAREDAT_HPLC_PIGMENTS')

Before converting to xarray, rename some columns and do some basic preprocessing

In [10]:
hplc.index.name = 'Id'

# Rename some columns
# new_names = {'day':'Day', 'month': 'Month', 'year': 'Year', 'lat': 'Lat', 'lon': 'Long'}
new_names = {'Day': 'day', 'Month': 'month', 'Year': 'year', 'Lat': 'lat', 'Long': 'lon'}
hplc = hplc.rename(columns=new_names)


# Remove entries with null date, longitude or latitude
valid_ind = hplc[['day', 'month', 'year', 'lat', 'lon']].isnull().prod(axis=1) ==0
hplc = hplc.loc[valid_ind]

# Add dates in a 'time' column
dates = pd.to_datetime(hplc[['month', 'day', 'year']])
hplc['time'] = dates

# Comvert to string some columns instead of object type
hplc['Event_num'] = hplc['Event_num'].astype(str)
hplc['Station'] = hplc['Station'].astype(str)
hplc['CTD'] = hplc['CTD'].astype(str)
hplc['Bottle_num']= hplc['Bottle_num'].astype(str)
hplc['Exp']= hplc['Exp'].astype(str)
hplc['Sample_num']= hplc['Sample_num'].astype(str)
hplc['Site']= hplc['Site'].astype(str)

# Replace empty values from non numerical ' ' to np.nan 
num_fields = ["Press (dbar)", "Merged_depth (m)","Number_of_pigments", "Total_Chla (ng/L)", "Total_Acc (ng/L)", "Total_Chla (mg/m3)", "Total_Acc (mg/m3)", "DVChla (ng/L)", "Chla (ng/L)", "Chla_ide (ng/L)", "Chla_allom (ng/L)", "Chla_prime (ng/L)", "Chlb (ng/L)", "DVChl b (ng/L)",    "Chlc (ng/L)",  "Chlc1_Chlc2_Mg_3_8_divinyl_pheoporphyrin_ a5 (ng/L)",  "Chlc1 (ng/L)", "Chlc1_like (ng/L)",    "Chlc2 (ng/L)", "Chlc1_Chlc2 (ng/L)",   "Chlc3 (ng/L)", "MgDVP (ng/L)", "19Hex (ng/L)", "19But (ng/L)", "Fucox (ng/L)", "Perid (ng/L)", "Prasino (ng/L)",   "Allox (ng/L)", "Lutein (ng/L)",    "Zeax (ng/L)",  "Zea_Lut  (ng/L)",  "Violax (ng/L)",    "Alpha_car (ng/L)", "Beta_car (ng/L)",  "Gamma_car (ng/L)", "Epsilon_car (ng/L)",   "Alpha_Beta_car (ng/L)",    "Neox (ng/L)",  "DD (ng/L)",    "DT (ng/L)",    "Viol_Neox (ng/L)", "Phaeopigments (ng/L)", "Phide_a (ng/L)",   "Phytin_a (ng/L)"]
for a in num_fields:
    hplc[a] = hplc[a].apply(lambda x: np.nan if x =='' or x == ' ' else x)

hplc_xr = hplc.to_xarray()

# xarray has problems with '/' symbol in variables, so change it
for a in hplc_xr:
    if '/' in a:
        a_renamed = re.sub(r'/', '*', a)
        hplc_xr = hplc_xr.rename({a:a_renamed})

In [11]:
# create columns with the mg*m^3 unit 
hplc_xr['chlide_a[mg*m^3]'] = hplc_xr['Chla_ide (ng*L)'] / 1000 

hplc_xr['chla[mg*m^3]'] =  hplc_xr['Chla (ng*L)'] / 1000

hplc_xr['chlb[mg*m^3]'] =  hplc_xr['Chlb (ng*L)'] / 1000

hplc_xr['chlc1+c2[mg*m^3]'] =  hplc_xr['Chlc1_Chlc2 (ng*L)'] / 1000

hplc_xr['fucox[mg*m^3]'] =  hplc_xr['Fucox (ng*L)'] / 1000

hplc_xr["19'hxfcx[mg*m^3]"] =  hplc_xr['19Hex (ng*L)'] / 1000

hplc_xr["19'btfcx[mg*m^3]"] =  hplc_xr['19But (ng*L)'] / 1000

hplc_xr["allox[mg*m^3]"] =  hplc_xr['Allox (ng*L)'] / 1000

hplc_xr["zeaxan[mg*m^3]"] =  hplc_xr['Zeax (ng*L)'] / 1000

hplc_xr["beta_car[mg*m^3]"] =  hplc_xr['Beta_car (ng*L)'] / 1000

hplc_xr["peridinin[mg*m^3]"] =  hplc_xr['Perid (ng*L)'] / 1000

hplc_xr["diatox[mg*m^3]"] =  hplc_xr['DT (ng*L)'] / 1000

hplc_xr["diadino[mg*m^3]"] =  hplc_xr['DD (ng*L)'] / 1000

In [15]:
# Save 
hplc_xr.to_netcdf(nc_out)


### Read ins-situ data 

In [16]:
# p_hplc_talone = Path('../../../data/processed/dataset_hplc_multi/y.nc')

In [17]:
# hplc_talone = xr.load_dataset(p_hplc_talone)

#### Finally, we read every archive and unify notation with 'pigment_names' constant

In [18]:
def get_all_archives_nc(path):
    archives = []
    for f in path.iterdir():
        if f.name == 'archive.nc':
            archives.append(f)
        elif f.is_dir():
            archives += get_all_archives_nc(f)
    return archives

archives = get_all_archives_nc(p_raw)

In [19]:
archives_nc = {arc.parent.name: xr.load_dataset(arc) for arc in archives}

In [20]:
archives_nc[list(archives_nc.keys())[4]]

<xarray.Dataset>
Dimensions:                 (Id: 32)
Coordinates:
  * Id                      (Id) int64 0 1 2 3 4 5 6 7 ... 25 26 27 28 29 30 31
Data variables: (12/56)
    identifier_product_doi  (Id) <U48 '10.5067_SeaBASS_TARA_OCEANS_POLAR_CIRC...
    received                (Id) float64 2.017e+07 2.017e+07 ... 2.01e+07
    investigators           (Id) <U55 'Herve_Claustre,Josephine_Ras,Emmanuel_...
    affiliations            (Id) <U119 'Laboratoire_d_Oceanographie_de_Villef...
    contact                 (Id) <U95 'claustre@obs-vlfr.fr,josephine.ras@obs...
    experiment              (Id) <U24 'TARA_Oceans_Polar_Circle' ... 'LOBO_ti...
    ...                      ...
    chl_b                   (Id) float64 0.0155 0.0438 0.018 ... 0.033 0.027
    tot_chl_b               (Id) float64 0.0155 0.0438 0.018 ... 0.033 0.027
    dv_chl_a                (Id) float64 -8.888e+03 -8.888e+03 ... 0.001 0.001
    chl_a                   (Id) float64 0.0936 0.1318 0.0859 ... 0.331 0.368
    tot_chl_a               (Id) float64 0.0936 0.1318 0.0859 ... 0.336 0.368
    phytin_a                (Id) float64 -8.888e+03 -8.888e+03 ... 0.043 0.053

In [21]:
# Methods to standardize string of pigment names
def search_high_match(xar, targets, thresh=0.8):
    for target in targets:
        closest_vars = get_close_matches(target, xar, 10, thresh)
        if len(closest_vars) != 0:
            return closest_vars[0]
    return None

def eliminate_invalid_pigment_indces(ds):
    # not_null_indices = np.array([~ds[pigment].isnull()  for pigment in pigment_names]).sum(axis=0).astype(bool)
    positive_indices = np.array([ds[pigment] >= 0       for pigment in pigment_names]).prod(axis=0).astype(bool)
    # small_indices = np.array([ds[pigment] < 5           for pigment in pigment_names]).prod(axis=0).astype(bool)
    # ds = ds.sel(Id=positive_indices*not_null_indices*small_indices)
    ds = ds.sel(Id=positive_indices)
    return ds

In [22]:
subs = {arch_name: [] for arch_name in archives_nc.keys()}
for pig_names in pigment_names_short:
    for arch_name, arch in archives_nc.items():
        subs[arch_name].append(search_high_match(arch, pig_names, 0.9))

pd.DataFrame(subs, index=pigment_names).T

,chlide_a[mg*m^3],chla[mg*m^3],chlb[mg*m^3],chlc1+c2[mg*m^3],fucox[mg*m^3],19'hxfcx[mg*m^3],19'btfcx[mg*m^3],diadino[mg*m^3],allox[mg*m^3],diatox[mg*m^3],zeaxan[mg*m^3],beta_car[mg*m^3],peridinin[mg*m^3]
BOWDOIN,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,None,perid
CALPOLY,chlide_a,None,None,None,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,beta-beta-car,perid
COLUMBIA_U,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,None,perid
LOV,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,None,perid
MAINE,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,None,allo,diato,zea,None,perid
MLI,None,chl_a,chl_b,None,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,beta-beta-car,perid
NASA_GSFC,chlide_a,tot_chl_a,None,None,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,None,perid
NOAA_CSC,None,None,None,None,fuco,hex-fuco,but-fuco,diadino,None,None,zea,None,perid
NOAA_NESDIS,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,beta-beta-car,perid
NRL,chlide_a,tot_chl_a,tot_chl_b,chl_c1c2,fuco,hex-fuco,but-fuco,diadino,allo,diato,zea,None,perid


In [23]:
# Perform substitutions on the datasets variables as table of matches above indicates

for arch_name, arch in archives_nc.items():
    renames = zip(subs[arch_name], pigment_names)
    renames = {key: val for key, val in renames if key is not None}
    archives_nc[arch_name] = arch.rename(renames)


In [24]:

# Rename also depth variable
for arch_name, arch in archives_nc.items():
    match = search_high_match(arch, ['depth', 'measurement_depth', 'Depth (m)'], 0.9)
    archives_nc[arch_name] = arch.rename({match: 'depth'})
    na = archives_nc[arch_name]['depth'] == 'NA'
    archives_nc[arch_name]['depth'][na] = '0.'
    archives_nc[arch_name]['depth'] = archives_nc[arch_name]['depth'].astype(float)

In [25]:
#  Concatenate all the archive datasets into one xarray
xar = xr.concat(archives_nc.values(), dim='Id', data_vars='minimal')
for var in ['Database_num', 'Cruise_des', 'F_TC', 'F_TA', 'F_TC_F_TA', 'F_Rat', 'F_Cr', 'Total_flags', 'Number_of_pigments']:
    if var in xar:
        xar[var] = xar[var].astype('float32')

# Concatenate D_ins hplc data
# xar = xr.concat([xar, hplc_talone], dim='Id')
xar = xar.assign_coords({'Id': range(len(xar.Id))})


In [26]:
# sample length so far
len(xar.Id)

53894

In [27]:
xar

<xarray.Dataset>
Dimensions:                                              (Id: 53894)
Coordinates:
  * Id                                                   (Id) int32 0 ... 53893
Data variables: (12/354)
    received                                             (Id) object '2014091...
    identifier_product_doi                               (Id) object '10.5067...
    investigators                                        (Id) object 'Collin_...
    affiliations                                         (Id) object 'Bowdoin...
    contact                                              (Id) object 'croesle...
    experiment                                           (Id) object 'ThreeRi...
    ...                                                   ...
    tpg_20filt_bincount                                  (Id) float64 nan ......
    dp_20filt                                            (Id) float64 nan ......
    dp_20filt_bincount                                   (Id) float64 nan ......
    number_of_data_rows                                  (Id) float64 nan ......
    instrument_manufacturer                              (Id) object nan ... nan
    instrument_model                                     (Id) object nan ... nan

In [28]:
final_hplc = xar

In [29]:
# xar.to_netcdf(p_raw / 'hplc.nc')
final_hplc.to_netcdf(p_pro / 'hplc.nc')

In [30]:
### Load back
hplc = xr.load_dataset(p_pro / 'hplc.nc')

In [31]:
hplc

<xarray.Dataset>
Dimensions:                                              (Id: 53894)
Coordinates:
  * Id                                                   (Id) int32 0 ... 53893
Data variables: (12/354)
    received                                             (Id) <U66 '20140912....
    identifier_product_doi                               (Id) <U39 '10.5067_S...
    investigators                                        (Id) <U38 'Collin_Ro...
    affiliations                                         (Id) <U15 'Bowdoin_C...
    contact                                              (Id) <U41 'croesler@...
    experiment                                           (Id) <U11 'ThreeRive...
    ...                                                   ...
    tpg_20filt_bincount                                  (Id) float64 nan ......
    dp_20filt                                            (Id) float64 nan ......
    dp_20filt_bincount                                   (Id) float64 nan ......
    number_of_data_rows                                  (Id) float64 nan ......
    instrument_manufacturer                              (Id) <U7 'nan' ... '...
    instrument_model                                     (Id) <U6 'nan' ... '...